# [STARTER] Exercise - Create an Agent with external API call enabled

In this exercise, you'll build an agent that can interact with external APIs to gather real-time data
and provide responses based on that information. You'll combine concepts from state management and
memory while adding the ability to make external API calls safely and effectively.


## Challenge

Your task is to create an agent that can make External API Calls:

- Implement tools that interact with real APIs
- Handle API responses and errors gracefully
- Use environment variables for API keys
- Process and format API data for user consumption

## Setup
First, let's import the necessary libraries:

In [1]:
import os
import random
from typing import List
import requests
from dotenv import load_dotenv

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [2]:
load_dotenv()

True

## Define API tools

Feel free to use any open service available through APIs.

Here are a few examples. You can follow the instructions given.
- https://jsonplaceholder.typicode.com/guide/
- https://www.exchangerate-api.com/
- https://openweathermap.org/

Or you can find one you're interested in here.
- https://github.com/public-apis/public-apis

In [3]:
# Define as many tools that access external APIs as you want
# Example:
@tool
def get_random_pokemon() -> dict:
    """Get a random Pokemon from the original 151"""
    URL = "https://pokeapi.co/api/v2/pokemon?limit=151"
    response = requests.get(URL)
    response.raise_for_status()
    return random.choice(response.json()['results'])

In [4]:
# Add all the tools you have defined
tools = [get_random_pokemon]

In [5]:
# Add instructions to your agent

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are an assistant that can help with: Identifying Pokemon\n"
    ),
    tools=tools
)

In [6]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agent

In [ ]:
query = "What are the different types of pokemon?"
session_id = "external_tools"

In [8]:
run1 = agent.invoke(
    query=query, 
    session_id=session_id,
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with: Identifying Pokemon
, tool_calls = None)
 -> (role = user, content = What are the different types of pokemon?, tool_calls = None)
 -> (role = assistant, content = There are 18 different Pokémon types. Here’s a list of them:

1. Normal
2. Fire
3. Water
4. Grass
5. Electric
6. Ice
7. Fighting
8. Poison
9. Ground
10. Flying
11. Psychic
12. Bug
13. Rock
14. Ghost
15. Dragon
16. Dark
17. Steel
18. Fairy

Each type has its strengths and weaknesses against other types, creating a complex system for battles and strategies in the Pokémon games., tool_calls = None)


## Check session histories

In [9]:
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])


# Run 1 {'run_id': 'fae325d5-3441-49cd-86bb-8d7ab1a7e718', 'start_timestamp': '2026-03-12 11:09:33.429361', 'end_timestamp': '2026-03-12 11:09:37.908574', 'snapshot_counts': 3}
Messages:
 -> (role = system, content = You are an assistant that can help with: Identifying Pokemon
, tool_calls = None)
 -> (role = user, content = What are the different types of pokemon?, tool_calls = None)
 -> (role = assistant, content = There are 18 different Pokémon types. Here’s a list of them:

1. Normal
2. Fire
3. Water
4. Grass
5. Electric
6. Ice
7. Fighting
8. Poison
9. Ground
10. Flying
11. Psychic
12. Bug
13. Rock
14. Ghost
15. Dragon
16. Dark
17. Steel
18. Fairy

Each type has its strengths and weaknesses against other types, creating a complex system for battles and strategies in the Pokémon games., tool_calls = None)


In [10]:
runs = agent.get_session_runs(session_id)
for run_object in runs:
    print(run_object)
    for snp in run_object.snapshots:
        print(f"-> {snp}")
    print("\n")

Run('fae325d5-3441-49cd-86bb-8d7ab1a7e718')
-> Snapshot('f636af45-b0c8-4d3b-8520-6bfafc9c0c1f') @ [2026-03-12 11:09:33.429481]: __entry__.State({'user_query': 'What are the different types of pokemon?', 'instructions': 'You are an assistant that can help with: Identifying Pokemon\n', 'messages': [], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('412728b7-0816-4c2f-98b4-721d8183103f') @ [2026-03-12 11:09:33.429775]: message_prep.State({'user_query': 'What are the different types of pokemon?', 'instructions': 'You are an assistant that can help with: Identifying Pokemon\n', 'messages': [SystemMessage(role='system', content='You are an assistant that can help with: Identifying Pokemon\n'), UserMessage(role='user', content='What are the different types of pokemon?')], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('e3aa1507-1a13-49da-94da-9142156c2238') @ [2026-03-12 11:09:37.908554]: llm_processor.State({'user_query': 'What are the diff